### **Using Chroma db locally*

In [1]:
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader

In [4]:
txt_loader = TextLoader(file_path='./docs/short-story.txt', autodetect_encoding=True)
loaded_txt = txt_loader.load()

print(loaded_txt[0].page_content[:200])

len(loaded_txt)

	Snow White and the Seven Dwarfs

Once upon a time in the middle of winter, when the flakes of
snow were falling like feathers from the sky, a queen sat at
a window sewing, and the frame of the window


1

In [5]:
ollama_embed = OllamaEmbeddings(model='mxbai-embed-large:latest', num_thread=3)

ollama_embed

OllamaEmbeddings(model='mxbai-embed-large:latest', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=3, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [7]:
rcts_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=50)
splitted_docs = rcts_splitter.split_documents(loaded_txt)

print(splitted_docs[4].page_content[:200])

splitted_docs[4].metadata

And as she was so beautiful the huntsman had pity on her and
said, run away, then, you poor child.  The wild beasts will soon
have devoured you, thought he, and yet it seemed as if a stone had
been ro


{'source': './docs/short-story.txt'}

In [9]:
embedded_docs = Chroma.from_documents(splitted_docs, ollama_embed)

embedded_docs

In [12]:
query = "who where the seven darfs?"

result = embedded_docs.similarity_search(query=query, k=3)

for data in result:
    print(data.page_content)

Then the first looked round and saw that there was a little
hollow on his bed, and he said, who has been getting into my
bed.  The others came up and each called out, somebody has been
lying in my bed too.  But the seventh when he looked at his bed
saw little snow-white, who was lying asleep therein.  And he
called the others, who came running up, and they cried out with
astonishment, and brought their seven little candles and let the
light fall on little snow-white.  Oh, heavens, oh, heavens, cried
they, what a lovely child.  And they were so glad that they did
not wake her up, but let her sleep on in the bed.  And the
seventh dwarf slept with his companions, one hour with each, and
so passed the night.
Then the first looked round and saw that there was a little
hollow on his bed, and he said, who has been getting into my
bed.  The others came up and each called out, somebody has been
lying in my bed too.  But the seventh when he looked at his bed
saw little snow-white, who was lying 

In [13]:
## - Saving the db
## Saving to the disk
## creates a sqlitedb - hosted any where

Chroma.from_documents(splitted_docs, ollama_embed, persist_directory='./chroma_db_new')


In [14]:
## load the saved db from disk
cdb_new = Chroma(persist_directory="./chroma_db_new", embedding_function=ollama_embed)
docs = cdb_new.similarity_search(query)
docs[0].page_content

'Then the first looked round and saw that there was a little\nhollow on his bed, and he said, who has been getting into my\nbed.  The others came up and each called out, somebody has been\nlying in my bed too.  But the seventh when he looked at his bed\nsaw little snow-white, who was lying asleep therein.  And he\ncalled the others, who came running up, and they cried out with\nastonishment, and brought their seven little candles and let the\nlight fall on little snow-white.  Oh, heavens, oh, heavens, cried\nthey, what a lovely child.  And they were so glad that they did\nnot wake her up, but let her sleep on in the bed.  And the\nseventh dwarf slept with his companions, one hour with each, and\nso passed the night.'

In [15]:
cdb_new.similarity_search_with_score(query=query)

[(Document(id='ad98ddaf-6fab-4642-81e8-1ee7dd4557c6', metadata={'source': './docs/short-story.txt'}, page_content='Then the first looked round and saw that there was a little\nhollow on his bed, and he said, who has been getting into my\nbed.  The others came up and each called out, somebody has been\nlying in my bed too.  But the seventh when he looked at his bed\nsaw little snow-white, who was lying asleep therein.  And he\ncalled the others, who came running up, and they cried out with\nastonishment, and brought their seven little candles and let the\nlight fall on little snow-white.  Oh, heavens, oh, heavens, cried\nthey, what a lovely child.  And they were so glad that they did\nnot wake her up, but let her sleep on in the bed.  And the\nseventh dwarf slept with his companions, one hour with each, and\nso passed the night.'),
  0.9373424053192139),
 (Document(id='dbc27fde-4ef8-47f7-a053-5c3c08b8ea8b', metadata={'source': './docs/short-story.txt'}, page_content='Little snow-white w